# QC confound 01 — Is the PBS signal specific to discarded cells?

The "discarded cells carry biology" framing makes a testable prediction: the *same*
PBS procedure applied to cells that QC **kept** should not classify ER+ vs Normal as
well as it does on the cells QC **discarded**. If it does, the classifier is reading a
specimen-wide QC property, and nothing about the discarded cells is special.

Every arm below runs the identical pipeline — per-specimen quantiles, same ISCB SCS
thresholds, same model — and differs only in which cells go in and how they are
featurized:

| Arm | Cells | Features | What it tests |
|---|---|---|---|
| `noise_pbs` | discarded | PBS one-hot | The published classifier (reference). |
| `retained_pbs` | retained | PBS one-hot | Negative control: same pipeline on kept cells. |
| `retained_pbs_matched` | retained, subsampled to the noise-cell count | PBS one-hot | As above, with equal cell numbers so quantile noise is comparable. |
| `all_pbs` | all | PBS one-hot | What a QC-agnostic pipeline would see. |
| `noise_qc` | discarded | raw QC metrics | Upper bound using sample-level shifts PBS cannot see. |
| `noise_qc_rank` | discarded | within-specimen QC ranks | Everything PBS *could* see; is PBS just a coarse summary of it? |
| `retained_qc` | retained | raw QC metrics | Do conditions differ in QC among cells nobody discards? |

**Protocol (shared by every arm):**
- Same specimens in every arm: those with >= 50 PBS-labelled noise cells, as published.
- Leave-one-**patient**-out folds; metrics per specimen.
- Significance: permute condition labels across specimens (never across cells) and
  re-run the whole evaluation.
- For continuity with the poster, the published protocol is also repeated on 100
  cell-balanced draws (6 Normal specimens, ER+ matched by cell count). The draws are
  shared, so arm comparisons are **paired**. `draws_mean_cell_accuracy` of `noise_pbs`
  is the reproduction of the published ~0.73.

Numbers will not match `10-a` exactly: this version seeds the draws, keeps both
patient-0029 specimens, holds out patients rather than specimens, and standardizes
features before the logistic regression.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from signals_in_the_noise.analysis.noise_phenotypes import ISCB_SCS_PBS_THRESHOLDS, PBS_LABELS
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.preprocessing.specimens import Cohort
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

POSITIVE, NEGATIVE = "ER+ tumour", "Normal"
SEED = 0
OUTPUT_DIR = get_data_path("processed") / "qc_confound"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gse = GSE161529()
cohort = Cohort.from_objects(gse.objects.values(), [POSITIVE, NEGATIVE])
cohort.overview().groupby("condition").agg(specimens=("n_cells", "size"), cells=("n_cells", "sum"))

In [ ]:
from signals_in_the_noise.modeling.datasets import standard_arms
from signals_in_the_noise.modeling.experiments import PUBLISHED_PROTOCOL, run_arm_suite

N_PERMUTATIONS = 200

arms = standard_arms(ISCB_SCS_PBS_THRESHOLDS)
pd.DataFrame(
    [(a.name, a.population, a.size_match, a.description) for a in arms],
    columns=["arm", "population", "size_match", "description"],
)

In [ ]:
result = run_arm_suite(
    cohort.obs,
    cohort.conditions,
    arms,
    positive_label=POSITIVE,
    negative_label=NEGATIVE,
    patients=cohort.patients,
    n_permutations=N_PERMUTATIONS,
    seed=SEED,
    **PUBLISHED_PROTOCOL,
)
print(f"{len(result.specimens)} specimens included")

## 1. Full-cohort results

`specimen_auc` and `specimen_balanced_accuracy` treat the specimen as the unit.
`permutation_p` is for `specimen_auc`; `permutation_null_mean` shows what chance looks
like under leave-one-out (it is often below 0.5).

In [ ]:
summary = result.summary()
summary[
    [
        "n_specimens",
        "n_cells",
        "specimen_auc",
        "specimen_balanced_accuracy",
        "permutation_null_mean",
        "permutation_null_q95",
        "permutation_p",
        "cell_accuracy",
        "draws_mean_cell_accuracy",
        "draws_mean_specimen_auc",
    ]
].round(3)

In [ ]:
from signals_in_the_noise.utils.visualization import plot_permutation_nulls

plot_permutation_nulls(result.permutations)
plt.show()

## 2. Paired comparison with the published arm

For each balanced draw, `mean_diff = noise_pbs − arm`. A positive difference with a
2.5–97.5 % interval above zero means discarded cells classify **better** than that arm.

In [ ]:
from signals_in_the_noise.utils.visualization import plot_arm_draws

order = [a.name for a in arms]
for metric in ("cell_accuracy", "specimen_auc"):
    print(metric)
    display(result.paired_vs_reference(metric).round(3))
    plot_arm_draws(result.draw_metrics, metric, order=order)
    plt.show()

## 3. What differs: PBS composition by condition

The per-cell model only sees which PBS subtype a cell falls in, so its prediction for a
specimen is a function of that specimen's PBS composition. Comparing compositions of
discarded vs retained cells shows *where* the separation comes from.

In [ ]:
from signals_in_the_noise.analysis.confounders import association_table
from signals_in_the_noise.modeling.datasets import specimen_composition
from signals_in_the_noise.utils.visualization import plot_covariates_by_condition

compositions = {}
for name in ("noise_pbs", "retained_pbs_matched", "all_pbs"):
    compositions[name] = specimen_composition(result.tables[name])
    association = association_table(
        compositions[name], positive=POSITIVE, negative=NEGATIVE, columns=list(PBS_LABELS)
    )
    print(name)
    display(association.round(4))
    plot_covariates_by_condition(
        compositions[name], list(PBS_LABELS), association=association, num_cols=3
    )
    plt.suptitle(name, y=1.02)
    plt.show()

## 4. Optional: random forest

The poster figure used a random forest. Set `RUN_RANDOM_FOREST = True` to repeat the
core arms with it (slower).

In [ ]:
from signals_in_the_noise.modeling.datasets import CORE_ARM_NAMES
from signals_in_the_noise.modeling.evaluation import random_forest_factory

RUN_RANDOM_FOREST = False
if RUN_RANDOM_FOREST:
    rf_result = run_arm_suite(
        cohort.obs,
        cohort.conditions,
        standard_arms(ISCB_SCS_PBS_THRESHOLDS, names=CORE_ARM_NAMES),
        positive_label=POSITIVE,
        negative_label=NEGATIVE,
        patients=cohort.patients,
        n_permutations=50,
        model_factory=random_forest_factory(),
        seed=SEED,
        **PUBLISHED_PROTOCOL,
    )
    display(rf_result.summary().round(3))
    display(rf_result.paired_vs_reference("cell_accuracy").round(3))
    rf_result.summary().to_csv(OUTPUT_DIR / "01_random_forest_summary.csv")

In [ ]:
summary.to_csv(OUTPUT_DIR / "01_arm_summary.csv")
result.draw_metrics.to_csv(OUTPUT_DIR / "01_draw_metrics.csv", index=False)
for metric in ("cell_accuracy", "specimen_auc"):
    result.paired_vs_reference(metric).to_csv(OUTPUT_DIR / f"01_paired_{metric}.csv")
pd.DataFrame({name: r.null for name, r in result.permutations.items()}).to_csv(
    OUTPUT_DIR / "01_permutation_nulls.csv", index=False
)
for name, composition in compositions.items():
    composition.to_csv(OUTPUT_DIR / f"01_composition_{name}.csv")

## Reading the results

| Observation | Meaning |
|---|---|
| `noise_pbs` significant **and** clearly above `retained_pbs_matched` (paired interval above 0) | Signal is concentrated in the discarded cells. Necessary for the framing but **not sufficient**: per-sample thresholds alone can produce this pattern (see below), so the framing holds only if it also survives notebook 03. |
| `retained_pbs(_matched)` about as good as `noise_pbs` | The PBS signal is a specimen-wide QC property. The "discarded cells" framing is **not** supported, even if PBS predicts condition. |
| `retained_qc` strong | Conditions differ in QC distributions of kept cells: the technical difference exists. Not decisive alone, because PBS removes location shifts. |
| `noise_qc_rank` about as good as `noise_pbs` | PBS is a coarse summary of within-specimen QC covariance; the subtype boundaries add nothing. |
| `noise_qc` much better than `noise_qc_rank` | Most available separation is in sample-level shifts, which PBS discards by design. |
| `noise_pbs` not significant under the specimen permutation | The published cell-level accuracy reflects pseudo-replication, not a specimen-level signal. |

**What these arms cannot rule out.** Every arm here uses Pal's per-sample thresholds to
decide which cells are noise. If ER+ and Normal specimens got different thresholds, the
*set* of discarded cells differs by condition even when the cells themselves do not, and
`noise_pbs` can separate conditions while every retained-cell arm stays near chance.
The unit test `test_threshold_driven_noise_membership_mimics_a_discarded_cell_signal`
builds exactly that world. Notebook 03 re-flags noise with one rule for all specimens;
that is the decisive test of this explanation.